# 40 - WGFMU programmed waveform + measured output + result viz

用途: 边跑边看 (a) 脚本构建的施加波形对不对(脉冲顺序/电压/脉宽/delay位置), (b) 测量输出 Id/Ig, (c) MW vs delay 测试数据图.

重要口径: 本 notebook 画的施加波形是程序下发的理想阶梯波(设 +-4V 就画 +-4V 方块), 用于确认脚本波形逻辑.
不是示波器实测的探针端波形 -- 无示波器, 软件证明不了探针端真实幅值/极性/脉宽.

安全: dry-run(AuditBackend) 绝不碰硬件; live 必须把 RUN_LIVE=True 显式打开.

关键约束: 每个 shot 开头 backend.clear() 会清空波形向量. 画波形只能针对单个 shot, 跑完该 shot 立即取 backend._patterns.

## Cell 1 - 环境 + 参数配置(改这里)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'src' / 'fefetlab').exists():
    if ROOT.parent == ROOT:
        break
    ROOT = ROOT.parent
SRC = ROOT / 'src'
assert (SRC / 'fefetlab').exists(), f'cannot find src/fefetlab, ROOT={ROOT}'
sys.path.insert(0, str(SRC))
sys.path.insert(0, str(ROOT / 'scripts'))
print('ROOT =', ROOT)

import wgfmu_next_round_minimal as M
from fefetlab.measurements.wgfmu.audit_backend import AuditBackend

# ====== tunable params (edit here) ======
STATE     = 'ERS'     # 'ERS' or 'PGM'
WRITE_V   = 4.0       # write magnitude V (ERS=+|v| / PGM=-|v|); last night +-5V near-breakdown, try 4
T_WRITE_S = 100e-6    # write width s
DELAY_S   = 1e-5      # write-to-read delay s (one representative delay for single-shot waveform)
VD_READ   = 0.05      # read drain V
VG_READS  = [-1.0, -0.7, -0.4, -0.2, 0.0, 0.2]   # read points (incl main -1.0V)
GATE_CH   = 202
DRAIN_CH  = 201
# ========================================

C_ERS = '#2659AD'; C_PGM = '#B80000'; C_READ = '#1A801A'; C_AXIS = '#4D4D4D'
plt.rcParams['font.family'] = 'serif'
print('params ready: STATE=%s WRITE_V=%+.1fV T_WRITE=%gs DELAY=%gs' % (STATE, WRITE_V, T_WRITE_S, DELAY_S))

## Cell 2 - build & plot programmed waveform (dry-run, no hardware)

用 AuditBackend 跑一个 shot, 从 _patterns 提取 gate/drain 向量画阶梯波. 先确认波形逻辑, 再决定上不上硬件.

In [ ]:
def vectors_to_xy(init_v, vectors):
    t = [0.0]; v = [init_v]; cur = 0.0
    for dt, volt in vectors:
        v.append(v[-1]); t.append(cur)
        cur += dt
        v.append(volt); t.append(cur)
    return np.array(t), np.array(v)

def build_one_shot(state, write_v, t_write, delay_s, vg_reads, vd_read):
    b = AuditBackend(gate_ch=GATE_CH, drain_ch=DRAIN_CH, channels=[201, 202, 301, 302])
    b.open_session('DRYRUN')
    v_write = +abs(write_v) if state == 'ERS' else -abs(write_v)
    rows = M.run_e1_shot(b, state=state, delay_s=delay_s, vg_reads=vg_reads,
                         vd_read=vd_read, n_pts=M.N_PTS, v_write=v_write, t_write=t_write)
    return b, rows

bk, win = build_one_shot(STATE, WRITE_V, T_WRITE_S, DELAY_S, VG_READS, VD_READ)
gp = bk._patterns['gp']; dp = bk._patterns['dp']
tg, vgw = vectors_to_xy(gp['init_v'], gp['vectors'])
td, vdw = vectors_to_xy(dp['init_v'], dp['vectors'])

fig, (axg, axd) = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
col = C_ERS if STATE == 'ERS' else C_PGM
axg.plot(tg * 1e3, vgw, color=col, lw=1.3)
axg.set_ylabel('VG (V)', fontsize=12); axg.grid(alpha=0.2)
axg.set_title('Programmed waveform (ideal step), NOT oscilloscope-measured  |  state=%s' % STATE, fontsize=10)
axd.plot(td * 1e3, vdw, color=C_READ, lw=1.3)
axd.set_ylabel('VD (V)', fontsize=12); axd.set_xlabel('time (ms)', fontsize=12); axd.grid(alpha=0.2)
plt.tight_layout(); plt.show()
print('gate segs:', len(gp['vectors']), ' drain segs:', len(dp['vectors']), ' read rows:', len(win))

## Cell 3 - LIVE run (needs explicit confirm)

RUN_LIVE 默认 False. 改 True 才真上硬件. live 用 RealWgfmuBackend 跑完整 stage, 保留 stop-gate.
建议先用 scripts/wgfmu_next_round_minimal.py 的 CLI 跑 live 更安全(有完整 stop-gate 流程); 本 cell 仅作 notebook 内 live 占位/示范.

In [ ]:
RUN_LIVE = False   # <<< keep False for dry-run; set True ONLY when ready to drive hardware

if not RUN_LIVE:
    print('RUN_LIVE=False -> skip hardware. (dry-run mode)')
else:
    # 推荐用 CLI 跑 live (完整 stop-gate); 此处给最小示范
    import subprocess
    cmd = [sys.executable, str(ROOT / 'scripts' / 'wgfmu_next_round_minimal.py'),
           '--stage', 'E1', '--live', '--confirm', 'E1',
           '--device-id', 'NB_LIVE', '--geometry', 'L10W10',
           '--e1-reps', '1', '--e1-wide-vg', '--e1-full-delays',
           '--write-v', str(WRITE_V)]
    print('RUN:', ' '.join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-3000:])
    print('STDERR tail:', r.stderr[-500:])

## Cell 4 - result figure from a real CSV (MW vs delay + per-branch)

读一份已有真实 E1 CSV, 画 MW=Id(ERS)-Id(PGM) @ 主读点, 并分态画 Id_ERS / Id_PGM 两条 branch.
改 CSV_PATH 指向你要看的那次 run 的 csv.

In [ ]:
# edit this to point at the CSV you want to inspect
CSV_PATH = r'D:/test/B1500/runs/dry'   # placeholder; set to a real e1_*.csv
MAIN_VG = -1.0

import glob, os
cand = CSV_PATH
if os.path.isdir(CSV_PATH):
    hits = glob.glob(os.path.join(CSV_PATH, '**', 'e1_*.csv'), recursive=True)
    cand = hits[0] if hits else None
    print('auto-picked CSV:', cand)

if not cand or not os.path.isfile(cand):
    print('no CSV found; set CSV_PATH to a real e1 csv path and rerun this cell')
else:
    df = pd.read_csv(cand)
    sub = df[np.isclose(df['Vg_read_V'].astype(float), MAIN_VG)]
    g = sub.groupby(['delay_s', 'state_target'])['Id_mean_A'].mean().unstack('state_target')
    g = g.sort_index()
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
    if 'ERS' in g: a1.semilogx(g.index, g['ERS'] * 1e6, 'o-', color=C_ERS, label='Id_ERS')
    if 'PGM' in g: a1.semilogx(g.index, g['PGM'] * 1e6, 's-', color=C_PGM, label='Id_PGM')
    a1.set_ylabel('Id (uA)'); a1.legend(); a1.grid(alpha=0.2)
    a1.set_title('per-branch Id vs delay @ Vg=%.1fV  (%s)' % (MAIN_VG, os.path.basename(cand)), fontsize=9)
    if 'ERS' in g and 'PGM' in g:
        mw = (g['ERS'] - g['PGM']) * 1e6
        a2.semilogx(mw.index, mw.values, '^-', color=C_READ, label='MW=Id(ERS)-Id(PGM)')
        a2.axhline(0, color=C_AXIS, lw=0.6)
        a2.set_ylabel('MW (uA)'); a2.legend(); a2.grid(alpha=0.2)
    a2.set_xlabel('write-to-read delay (s)')
    plt.tight_layout(); plt.show()